# 4교시. 필요한 값만 JSON으로 담기

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leecks1119/document_ai_lecture/blob/document_ai_lecture_2026/colab/04_genai_extraction.ipynb)

**목표:** 스키마에 맞춰 값을 구조화하고 없는 값은 null로 처리합니다.

**결과물:** `receipt.json`

- 기본 경로는 API 키와 OCR 모델 다운로드가 필요 없습니다.
- 선택 실습은 기본값이 `False`입니다.
- 실제 개인정보가 없는 합성 영수증만 사용합니다.


In [ ]:
import platform
import sys
from pathlib import Path

OUTPUT_DIR = Path("course_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Output:", OUTPUT_DIR.resolve())


In [ ]:
import json
import re

SAMPLE_OCR_TEXT = '샘플문구점\n거래일자: 2026-07-27\n연필 2개 × 1,000원 = 2,000원\n노트 1개 × 3,000원 = 3,000원\n합계: 5,000원\n'
SAMPLE_RECEIPT = {'document_type': 'receipt', 'store_name': '샘플문구점', 'date': '2026-07-27', 'total_amount': 5000, 'items': [{'name': '연필', 'quantity': 2, 'unit_price': 1000, 'line_total': 2000}, {'name': '노트', 'quantity': 1, 'unit_price': 3000, 'line_total': 3000}], 'source_mode': 'mock'}


## 핵심 3개

1. 스키마는 필드명·자료형·필수 여부를 정합니다.
2. 원문에 없는 값은 `null`입니다.
3. JSON 문법과 값의 사실성은 별도 검사입니다.


In [ ]:
RECEIPT_SCHEMA = {
    "date": {"type": ["string", "null"]},
    "total_amount": {"type": ["integer", "null"], "minimum": 0},
}


def to_int(value):
    return int(value.replace(",", ""))


def mock_extract(ocr_text):
    lines = [line.strip() for line in ocr_text.splitlines() if line.strip()]
    date_match = re.search(r"\d{4}-\d{2}-\d{2}", ocr_text)
    total_match = re.search(r"합계\s*:\s*([\d,]+)원", ocr_text)
    item_pattern = re.compile(
        r"(?P<name>.+?)\s+(?P<quantity>\d+)개\s*[×x]\s*"
        r"(?P<unit>[\d,]+)원\s*=\s*(?P<line>[\d,]+)원"
    )
    items = []
    for line in lines:
        match = item_pattern.search(line)
        if match:
            items.append({
                "name": match.group("name").strip(),
                "quantity": int(match.group("quantity")),
                "unit_price": to_int(match.group("unit")),
                "line_total": to_int(match.group("line")),
            })
    return {
        "document_type": "receipt",
        "store_name": lines[0] if lines else None,
        "date": date_match.group(0) if date_match else None,
        "total_amount": to_int(total_match.group(1)) if total_match else None,
        "items": items,
        "source_mode": "mock",
    }


## 실습. mock JSON을 만들고 원문과 대조


In [ ]:
receipt = mock_extract(SAMPLE_OCR_TEXT)

assert receipt["date"] == "2026-07-27"
assert receipt["total_amount"] == 5000
assert len(receipt["items"]) == 2

output_path = OUTPUT_DIR / "receipt.json"
output_path.write_text(
    json.dumps(receipt, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
print(json.dumps(receipt, ensure_ascii=False, indent=2))


## 생성형 AI용 추출 프롬프트

실제 API 없이도 역할·목표·제약·출력 형식을 작성하는 연습을 합니다.


In [ ]:
EXTRACTION_PROMPT = f'''역할: 영수증 정보 추출 도우미
목표: 상호명, 날짜, 품목, 합계를 JSON으로 추출
제약조건:
- 원문에 없는 값은 null
- JSON 외 설명 금지

OCR 텍스트:
{SAMPLE_OCR_TEXT}'''

print(EXTRACTION_PROMPT)


## 선택 확인. API 연결 전 준비사항

실제 API를 호출하지 않습니다. 비밀 저장 방식과 조직의 데이터 처리 조건을
확인하는 항목이며, 기본 실습은 mock 결과로 이미 완료됐습니다.


In [ ]:
CHECK_OPTIONAL_API_READINESS = False

if CHECK_OPTIONAL_API_READINESS:
    print(
        "실제 호출은 하지 않습니다. Colab Secrets, 조직 승인, "
        "데이터 처리 조건을 확인하세요."
    )
else:
    print("API 연결 준비 확인을 건너뛰고 mock 결과로 완료했습니다.")


## 확인

- 없는 값 규칙이 `null`인가?
- 합계가 숫자 `5000`인가?
- JSON 모양과 원문 근거를 따로 확인했는가?
